# SVM Lineal — Riesgo cardiovascular (BRFSS 2024)

Este cuaderno implementa el modelo SVM Lineal. Se emplean técnicas de validación cruzada, imputación proporcional, normalización y SMOTE.

Los datos de entrada ya vienen imputados, normalizados y balanceados con SMOTENC, generados una sola vez por scripts/preprocesar_pliegues.py (ver README del proyecto). Si la carpeta data_procesada/ no existe, es necesario correr ese script antes de continuar.


In [7]:
import os
import sys
sys.path.append("..")

import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

from scripts import pipeline as pl

DATA_DIR = "../data_procesada"
N_PLIEGUES = 5


def cargar(nombre):
    """Lee una particion ya preprocesada (generada por scripts/preprocesar_pliegues.py)."""
    tabla = pd.read_parquet(f"{DATA_DIR}/{nombre}")
    X = tabla.drop(columns=[pl.TARGET_COL])
    y = tabla[pl.TARGET_COL]
    return X, y

In [3]:
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        "No se encontro data_procesada/. Corre primero 'python scripts/preprocesar_pliegues.py' "
        "desde la raiz del proyecto (tarda unos 15 minutos, se corre una sola vez por maquina)."
    )

print("cache encontrado en", DATA_DIR)

cache encontrado en ../data_procesada


## Prueba sobre el pliegue 0
Antes de ejecutar el ciclo completo de 5 pliegues, se valida el procedimiento sobre un único pliegue, con el fin de confirmar que el modelo entrena correctamente sobre estos datos y que la función pipeline.evaluar() procesa sus predicciones sin errores.

In [4]:
X_train, y_train = cargar("fold_0_train.parquet")
X_test, y_test = cargar("fold_0_test.parquet")
prot_test = X_test[pl.PROTECTED_COLS]

print("train:", X_train.shape, "| balance:", y_train.value_counts().to_dict())

train: (634536, 30) | balance: {0: 317268, 1: 317268}


In [5]:


modelo = CalibratedClassifierCV(estimator=LinearSVC(
        C=1.0,
        random_state=pl.RANDOM_STATE,
        dual=False,
    ),
    ensemble=False
)

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

In [6]:
resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)

print("RESUMEN (pliegue 0):")
for k, v in resumen.items():
    print(f"  {k}: {v:.4f}")

detalle["_RACE"]

RESUMEN (pliegue 0):
  auroc: 0.7882
  eod_sexo: 0.0903
  fpr_disp_sexo: 0.0409
  calib_desv_sexo: 2.3079
  eod_raza: 0.5612
  fpr_disp_raza: 0.2604
  calib_desv_raza: 3.1295
  eod_ingreso: 0.2771
  fpr_disp_ingreso: 0.2488
  calib_desv_ingreso: 3.0088


,grupo,n,tpr,fpr,calibracion
0,1.0,66555,0.730519,0.292510,3.070906
1,2.0,6987,0.590613,0.204180,2.922636
2,4.0,2469,0.380531,0.056027,4.129540
3,7.0,2053,0.169355,0.032133,1.307400
4,8.0,9544,0.381032,0.076820,3.243482
5,6.0,701,0.300000,0.057283,1.716560
6,3.0,1248,0.491228,0.142061,2.211814
7,5.0,395,0.319149,0.066092,1.715392
